# Lab 08 · Reading diagrams with a vision model

**Day 3 · S16, lab 1 of 3** · Budget: 20 min of the 25 min slot · Runs on: Colab with a T4 GPU (a laptop works, slower)

A vision model reads 10 architecture and process diagrams and returns each one as JSON: the boxes, and the arrows between them. You score that JSON against ground truth and find out where it breaks.

### Where does the image go?

In Colab the image leaves your laptop either way, so the question isn't laptop versus cloud. It's **self-hosted versus vendor API**:

| | Self-hosted model | Vendor API |
|---|---|---|
| Runs on | a machine you control. Here the Colab runtime stands in for OQ's own Azure VM | the vendor's servers |
| The image | stays inside your boundary | goes to a third party |
| In this lab | Ollama running `qwen2.5vl:3b` | OpenAI `gpt-4.1-mini` |

A self-hosted model on OQ's own VM is what OQ would actually deploy for drawings that can't leave the network. You run the same task on both and compare them on the same score.

### The image set

Every image is synthetic, drawn by `scripts/render_images.py`, so the ground truth is exact.

| Tier | Images | What it tests |
|---|---|---|
| 1 | dia_01 to dia_03 | clean, 4 boxes |
| 2 | dia_04 to dia_06 | dense: 8 or 9 boxes, labelled arrows, some lines cross |
| 3 | dia_07 to dia_09 | **the same diagrams as tier 2**, photographed from a printout |
| known-bad | dia_10 | clean, but built to trip the model |

## 1. Setup

On a fresh Colab runtime the next two cells install Ollama and download the self-hosted model, which takes about 3 minutes. Start them now and read ahead while they run.

The vendor model needs an `OPENAI_API_KEY`: in Colab, add it under **Secrets** (the key icon on the left); on a laptop, put it in a `.env` file in the lab folder. If a model isn't available the lab carries on with the other one, or with the facilitator's saved outputs.

In [ ]:
# Setup: find the lab folder, detect the runtime, install pinned packages on Colab.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = os.environ.get("LAB_REPO_URL", "")  # Colab: the course repo URL, once it is published


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "vision_client.py").exists():
            return p


ROOT = find_root()
if ROOT is None and IN_COLAB and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    ROOT = Path("/content/lab")
if ROOT is None:
    raise RuntimeError("Lab folder not found. Open this notebook from inside it, or set LAB_REPO_URL on Colab.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.0.0", "python-dotenv==1.1.0"], check=True)
sys.path.insert(0, str(ROOT / "scripts"))
print("lab folder:", ROOT, "| runtime:", "Colab" if IN_COLAB else "local")

In [ ]:
from vision_client import ensure_ollama, load_openai_key, prebaked_models, run_batch, self_hosted, vendor_api

RUN_MODE = os.environ.get("LAB_RUN_MODE", "live")  # "live" calls the models, "prebaked" replays a saved run
LAB = "08"
OUT = ROOT / "outputs" / LAB
PREBAKED = Path(os.environ.get("LAB_PREBAKED_DIR", ROOT / "facilitator" / "prebaked_outputs")) / LAB

MODELS = []
if RUN_MODE == "live":
    try:
        ensure_ollama()
        MODELS.append(self_hosted())
    except Exception as e:
        print("self-hosted model unavailable:", e)
    if load_openai_key(ROOT):
        MODELS.append(vendor_api())
    else:
        print("vendor API unavailable: no OPENAI_API_KEY in Colab secrets or .env")
if not MODELS:
    MODELS = prebaked_models(PREBAKED)
    print("using prebaked outputs from", PREBAKED)
assert MODELS, f"No live model and no prebaked outputs in {PREBAKED}. Ask the facilitator."
print("models:", [m.label for m in MODELS])

## 2. Look at the images

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from render_images import build_image_set

MANIFEST = ROOT / "corpus" / "images" / "manifest.csv"
TRUTH = ROOT / "data" / "eval" / "image_ground_truth"
if not MANIFEST.exists():
    build_image_set(ROOT)
manifest = pd.read_csv(MANIFEST)
items = manifest[manifest.kind == "diagram"]


def show(item_ids, height=4.5):
    fig, axes = plt.subplots(1, len(item_ids), figsize=(6.5 * len(item_ids), height))
    for ax, item_id in zip(np.atleast_1d(axes), item_ids):
        ax.imshow(Image.open(ROOT / "corpus" / "images" / f"{item_id}.png"))
        ax.set_title(item_id)
        ax.axis("off")
    plt.show()


def truth(item_id):
    return json.loads((TRUTH / f"{item_id}.json").read_text())


items[["item_id", "tier", "degradation", "source", "known_bad"]]

In [ ]:
show(["dia_01", "dia_05", "dia_08"])

dia_05 and dia_08 have the same content; only the image quality differs. Any score gap between tier 2 and tier 3 comes from the photo, not the diagram.

## 3. The extraction contract

You don't ask the model to "describe the diagram". You give it a schema, so the output can go straight into an integration catalogue or a CMDB, and so it can be scored. Both backends enforce the schema while they generate, so the output is always valid JSON. Valid JSON isn't the same as correct JSON; the score in section 6 checks that.

In [ ]:
DIAGRAM_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": ["string", "null"]},
        "nodes": {"type": "array", "items": {"type": "string"}},
        "edges": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "from": {"type": "string"},
                    "to": {"type": "string"},
                    "label": {"type": ["string", "null"]},
                },
                "required": ["from", "to", "label"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["title", "nodes", "edges"],
    "additionalProperties": False,
}

PROMPT = """You are reading an engineering or IT architecture diagram.
Return every box as a node, using the exact text written in the box.
Return every arrow as an edge: from the box at the tail to the box the arrowhead points at,
with the text written on the arrow as its label (null if the arrow has no text).
Only report what is drawn. Do not add connections you would expect but cannot see."""

## 4. One image, one model

In [ ]:
from score_extraction import score_diagram

first = run_batch(MODELS[0], [{"item_id": "dia_01", "image": ROOT / "corpus/images/dia_01.png"}],
                  prompt=PROMPT, schema=DIAGRAM_SCHEMA, out_dir=OUT, prebaked_dir=PREBAKED)[0]
print(json.dumps(first["output"], indent=2))
score_diagram(first["output"], truth("dia_01"))

## 5. All ten diagrams, both models

Results are cached in `outputs/08/<model>/`, so running this cell again after the break costs nothing. The vendor API takes 4 requests at a time; the self-hosted model takes one at a time on the GPU.

In [ ]:
jobs = [{"item_id": r.item_id, "image": ROOT / r.image} for r in items.itertuples()]
for vm in MODELS:
    run_batch(vm, jobs, prompt=PROMPT, schema=DIAGRAM_SCHEMA, out_dir=OUT, prebaked_dir=PREBAKED,
              workers=4 if vm.backend == "openai" else 1)

## 6. Score

`scripts/score_extraction.py` matches boxes by name, ignoring case and punctuation. It also accepts a bare tag, so `V-201` matches `Inlet Separator V-201`. Then it checks each arrow, including its direction. **score** is the average of node F1 and edge F1, from 0 to 1.

In [ ]:
from score_extraction import score_dir, summary

scores = score_dir(OUT, TRUTH, manifest, "diagram")
summary(scores)

In [ ]:
scores[["item_id", "tier", "label", "score", "node_f1", "edge_f1", "edge_label_acc",
        "invented_edges", "reversed_edges", "seconds"]]

### Same diagram, worse photo

Tier 2 and tier 3 hold the same three diagrams. Which metric drops first: node F1, edge F1, or the arrow labels? Box text is large and arrow labels are small, and a blurry photo loses small text first.

In [ ]:
pairs = {"dia_04": "dia_07", "dia_05": "dia_08", "dia_06": "dia_09"}
cols = ["node_f1", "edge_f1", "edge_label_acc"]
clean = scores[scores.item_id.isin(pairs.keys())].groupby("label")[cols].mean()
photo = scores[scores.item_id.isin(pairs.values())].groupby("label")[cols].mean()
pd.concat({"tier 2 (clean)": clean, "tier 3 (photo)": photo}, axis=1).round(2)

## 7. The known-bad case

dia_10 is clean and small, and a careful person can read every arrow. It has three traps:

- two lines cross in the middle with no junction
- two pumps differ by one letter: P-101A and P-101B
- the bottom arrow points right to left, against the usual reading direction

In the errors below, look for an **invented edge** (the model followed the wrong line through the crossing), a **reversed edge** (it assumed left to right), and a missing or merged pump.

In [ ]:
show(["dia_10"], height=6)
for r in scores[scores.known_bad].itertuples():
    print(f"{r.label}: score {r.score}")
    for e in r.errors:
        print("   ", e)

In an integration catalogue, a wrong connection is worse than a missing one, because nobody double-checks a connection that looks right. So treat extraction output as a draft: a person reviews it, or it's checked against a system of record, before anything acts on it.

## 8. Try it: a better prompt (if you have time)

Change the prompt and run it again on the known-bad case only. The version below makes the model trace each line before it lists the edges. Output goes to `outputs/08_try/`, so it doesn't mix with the main run. Edit `PROMPT_V2` and re-run to try your own ideas.

In [ ]:
PROMPT_V2 = PROMPT + """
Before listing edges, trace each line from one end to the other. Where two lines cross, stay on the same straight line.
An edge goes from the end with no arrowhead to the end with the arrowhead, whichever way that points on the page."""

TRY_OUT, TRY_PREBAKED = ROOT / "outputs" / "08_try", PREBAKED.parent / "08_try"
known_bad = [{"item_id": "dia_10", "image": ROOT / "corpus/images/dia_10.png"}]
for vm in MODELS:
    rec = run_batch(vm, known_bad, prompt=PROMPT_V2, schema=DIAGRAM_SCHEMA, out_dir=TRY_OUT,
                    prebaked_dir=TRY_PREBAKED, force=True, verbose=False)[0]
    before = scores[(scores.item_id == "dia_10") & (scores.model == vm.name)].score.iloc[0]
    after = score_diagram(rec["output"], truth("dia_10"))
    print(f"{rec['label']}: score {before} -> {after['score']}")
    for e in after["errors"]:
        print("   ", e)

## 9. What to take away

- **A schema guarantees valid JSON, not correct JSON.** You only know how correct it is if you have ground truth to score against.
- **Test on your own images, in tiers.** A model that aces clean diagrams can fall apart on a phone photo of the same drawing.
- **Keep a known-bad case in your eval set,** so you know *how* the model fails, not only how often.
- **Self-hosted versus vendor API is a data-classification decision first.** Compare the scores only once you know which one you're allowed to use.

## Facilitator: save this run as the room's fallback

In [ ]:
from vision_client import promote_to_prebaked

PROMOTE = False  # facilitator only: after a good live run, keep it for when the network or a model fails
if PROMOTE and RUN_MODE == "live":
    promote_to_prebaked(OUT, PREBAKED)
    promote_to_prebaked(TRY_OUT, TRY_PREBAKED)